In [1]:
import urllib.request
import bs4 as bs
import nltk
import random
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Fetching data
def fetch_wiki_data(url):
    get_link = urllib.request.urlopen(url)
    get_link = get_link.read()
    data = bs.BeautifulSoup(get_link, 'lxml')
    data_paragraphs = data.find_all('p')
    data_text = ''
    for para in data_paragraphs:
        data_text += para.text
    return data_text.lower()

In [3]:
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anjal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\anjal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
wnlemmatizer = nltk.stem.WordNetLemmatizer()

In [5]:
def perform_lemmatization(tokens):
    return [wnlemmatizer.lemmatize(token) for token in tokens]

In [6]:
punctuation_removal = dict((ord(punctuation), None) for punctuation in string.punctuation)

In [7]:
def get_processed_text(document):
    return perform_lemmatization(nltk.word_tokenize(document.lower().translate(punctuation_removal)))

In [8]:
# Greeting function
greeting_input = ("hey", "hello", "good morning", "good evening", "morning", "evening", "hi")
greeting_response = ["hey", "hello", "how are you?", "hello, how are you doing?", "Welcome, I am good", "*nods*"]

In [9]:
def generate_greeting_response(greeting):
    for token in greeting.split():
        if token.lower() in greeting_input:
            return random.choice(greeting_response)
    return None

In [13]:
def generate_response(user_input, data_sentences):
    bot_response = ''
    data_sentences.append(user_input)

    word_vectorizer = TfidfVectorizer(tokenizer=get_processed_text, stop_words='english')
    all_word_vectors = word_vectorizer.fit_transform(data_sentences)
    similar_vector_values = cosine_similarity(all_word_vectors[-1], all_word_vectors)
    similar_sentence_number = similar_vector_values.argsort()[0][-2]

    matched_vector = similar_vector_values.flatten()
    matched_vector.sort()
    matched_vector = matched_vector[-2]

    # Set a threshold to prevent irrelevant responses
    if matched_vector < 0.3:  # You can tune this threshold
        bot_response = 'Could not find a matching answer!'
    else:
        bot_response = data_sentences[similar_sentence_number]

    data_sentences.remove(user_input)
    return bot_response

In [14]:
def chatbot():
    wiki_url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
    data_text = fetch_wiki_data(wiki_url)
    data_sentences = nltk.sent_tokenize(data_text)

    continue_dialog = True
    print('Chatbot: Ask me anything about AI!')
    
    while continue_dialog:
        human_text = input("You: ").lower()

        if human_text != 'bye':
            if human_text in ['thanks', 'thank you']:
                print("Chatbot: You're welcome!")
                continue_dialog = False
            else:
                greeting = generate_greeting_response(human_text)
                if greeting:
                    print(f"Chatbot: {greeting}")
                else:
                    print(f"Chatbot: {generate_response(human_text, data_sentences)}")
        else:
            continue_dialog = False
            print("Chatbot: Goodbye!")

In [ ]:
chatbot()

Chatbot: Ask me anything about AI!
You: hey
Chatbot: hello
You: what is artificial learning


C:\Users\anjal\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\anjal\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ha', 'le', 'u', 'wa'] not in stop_words.
  warnings.warn(


Chatbot: [47] deep learning is a type of machine learning that runs inputs through biologically inspired artificial neural networks for all of these types of learning.
You: artificial intelligenc
Chatbot: Could not find a matching answer!
You: what is artificial intelligence
Chatbot: 
artificial intelligence (ai), in its broadest sense, is intelligence exhibited by machines, particularly computer systems.
You: histrory
Chatbot: Could not find a matching answer!
You: history of artificial intelligence
Chatbot: Could not find a matching answer!
